In [86]:
import requests as rq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [87]:
url = "https://air-quality-api.open-meteo.com/v1/air-quality"

In [88]:
params = {
    "latitude": 16.694863,
    "longitude": 80.318784,
    "hourly": "pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,us_aqi",
    "start_date": "2025-12-31",   
    "end_date": "2026-11-01",     
    "timezone": "Asia/Kolkata",
}

In [89]:
data = rq.get(url, params)

In [90]:
print(data.status_code)

200


In [91]:
data_set = data.json()['hourly']

In [92]:
Air_Quality = pd.DataFrame(data_set)

In [93]:
Air_Quality.head()

,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,us_aqi
0,2025-12-31T00:00,95.9,94.1,571.0,10.0,14.8,83.0,2.0,177
1,2025-12-31T01:00,94.8,92.7,557.0,8.6,13.7,79.0,2.0,176
2,2025-12-31T02:00,95.9,93.6,553.0,7.6,12.8,76.0,2.0,176
3,2025-12-31T03:00,97.3,95.0,573.0,7.3,12.4,74.0,2.0,175
4,2025-12-31T04:00,100.6,97.8,602.0,7.4,12.2,74.0,2.0,175


In [94]:
Air_Quality.to_csv('Air_Quality_data_set.csv')

In [95]:
data_set_1 = pd.read_csv('/kaggle/working/Air_Quality_data_set.csv', parse_dates = ['time'], date_format = 'mixed')

In [96]:
data_set_1.head()

,Unnamed: 0,time,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,us_aqi
0,0,2025-12-31 00:00:00,95.9,94.1,571.0,10.0,14.8,83.0,2.0,177
1,1,2025-12-31 01:00:00,94.8,92.7,557.0,8.6,13.7,79.0,2.0,176
2,2,2025-12-31 02:00:00,95.9,93.6,553.0,7.6,12.8,76.0,2.0,176
3,3,2025-12-31 03:00:00,97.3,95.0,573.0,7.3,12.4,74.0,2.0,175
4,4,2025-12-31 04:00:00,100.6,97.8,602.0,7.4,12.2,74.0,2.0,175


In [97]:
data_set_1.rename(columns = {
    'time':'date',
    'Unnamed: 0':'index'
}, inplace = True)

In [98]:
data_set_1.head()

,index,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,us_aqi
0,0,2025-12-31 00:00:00,95.9,94.1,571.0,10.0,14.8,83.0,2.0,177
1,1,2025-12-31 01:00:00,94.8,92.7,557.0,8.6,13.7,79.0,2.0,176
2,2,2025-12-31 02:00:00,95.9,93.6,553.0,7.6,12.8,76.0,2.0,176
3,3,2025-12-31 03:00:00,97.3,95.0,573.0,7.3,12.4,74.0,2.0,175
4,4,2025-12-31 04:00:00,100.6,97.8,602.0,7.4,12.2,74.0,2.0,175


In [99]:
data_set_1.sort_values(by = 'date',  inplace = True)

In [100]:
data_set_1['Year'] = data_set_1['date'].dt.year
data_set_1['Month'] = data_set_1['date'].dt.month
data_set_1['Day'] = data_set_1['date'].dt.day

In [101]:
data_set_1.head()

,index,date,pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,sulphur_dioxide,ozone,dust,us_aqi,Year,Month,Day
0,0,2025-12-31 00:00:00,95.9,94.1,571.0,10.0,14.8,83.0,2.0,177,2025,12,31
1,1,2025-12-31 01:00:00,94.8,92.7,557.0,8.6,13.7,79.0,2.0,176,2025,12,31
2,2,2025-12-31 02:00:00,95.9,93.6,553.0,7.6,12.8,76.0,2.0,176,2025,12,31
3,3,2025-12-31 03:00:00,97.3,95.0,573.0,7.3,12.4,74.0,2.0,175,2025,12,31
4,4,2025-12-31 04:00:00,100.6,97.8,602.0,7.4,12.2,74.0,2.0,175,2025,12,31


In [102]:
X = data_set_1.drop(['index', 'date', 'us_aqi'], axis = 1)
y = data_set_1['us_aqi']

In [103]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

In [104]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)

In [105]:
model = RandomForestRegressor()

In [106]:
model.fit(X_train, y_train)

RandomForestRegressor()

In [107]:
model.score(X_train,y_train)

0.989429959262069

In [108]:
model.score(X_test, y_test)

0.9245536003003378

In [109]:
pred = model.predict(X_test)

In [110]:
pred

array([ 72.51,  62.11, 151.46, ...,  97.08,  62.28,  75.52])

In [111]:
results = pd.DataFrame({
    "actual": y_test.values,
    "predicted": pred,
})
results["error"] = (results["actual"] - results["predicted"])
        

In [112]:
results[results['error'] == 0]

,actual,predicted,error


In [113]:
results

,actual,predicted,error
0,72,72.51,-0.51
1,70,62.11,7.89
2,152,151.46,0.54
3,130,102.88,27.12
4,79,75.61,3.39
...,...,...,...
1464,145,117.99,27.01
1465,159,159.55,-0.55
1466,122,97.08,24.92
1467,59,62.28,-3.28


In [114]:
import pickle as pkl

In [115]:
with open ('RandomForest_aqi.pkl', 'wb') as file:
    pkl.dump(model, file)